# Greek Banking Sector — Quick Analysis

A short companion to the dashboard:
- **DuPont (3 steps):** ROE = net profit margin × asset yield × leverage, on average balances
- **Asset quality:** loan impairments, cost of risk and NPE ratio
- **NIM and efficiency**
- **Peer heatmap for 2024**

**Data:** `kpis_final.csv`, built by `pipeline/build_dataset.py` from the annual reports. The fuller analysis is in `03_analysis/`.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Load Data

In [2]:
# Load KPIs dataset
df = pd.read_csv('../data/processed/kpis_final.csv')
print(df[['bank', 'year', 'nii', 'net_profit', 'total_assets', 'roe', 'nim', 'cost_to_income', 'cet1', 'npe_ratio']].to_string(index=False))
print(f"\nShape: {df.shape}")

        bank  year    nii  net_profit  total_assets  roe  nim  cost_to_income  cet1  npe_ratio
  Alpha Bank  2022 1173.8       368.1       78011.4  6.0 1.55            48.9 13.20        7.8
  Alpha Bank  2023 1659.0       618.0       72421.0  9.1 2.21            38.3 14.40        6.0
  Alpha Bank  2024 1645.0       654.0       72075.0  8.5 2.28            39.0 16.30        3.8
    Eurobank  2022 1480.0      1347.0       81474.0 22.1 1.86            28.2 16.00        5.2
    Eurobank  2023 2174.0      1140.0       79781.0 15.7 2.70            31.4 16.90        3.5
    Eurobank  2024 2507.0      1448.0      101150.0 17.2 2.77            32.9 16.80        2.9
         NBG  2022 1369.0      1120.0       78113.0 18.4 1.69            36.3 16.60        5.2
         NBG  2023 2263.0      1106.0       74584.0 15.7 2.96            33.6 17.80        3.7
         NBG  2024 2356.0      1158.0       74957.0 14.4 3.15            36.5 18.30        2.6
Piraeus Bank  2022 1353.0       950.0       74645.

## 1. DuPont Analysis

**ROE = net profit margin × asset yield × leverage**

- **Net profit margin** = net profit / operating income
- **Asset yield** = operating income / average total assets
- **Leverage** = average total assets / average equity

Using averages makes the product equal the ROE in the dataset.

In [3]:
dupont = df.copy()
dupont['net_profit_margin'] = dupont['net_profit'] / dupont['operating_income'] * 100
dupont['asset_yield'] = dupont['operating_income'] / dupont['avg_total_assets'] * 100
dupont['financial_leverage'] = dupont['avg_total_assets'] / dupont['avg_equity']
dupont['roe_calculated'] = dupont['net_profit_margin'] * dupont['asset_yield'] * dupont['financial_leverage'] / 100

dupont_2024 = dupont[dupont['year'] == 2024][['bank', 'net_profit_margin', 'asset_yield', 'financial_leverage', 'roe_calculated', 'roe']].round(2)
print("DuPont Analysis — 2024:")
print(dupont_2024.to_string(index=False))
assert ((dupont['roe_calculated'] - dupont['roe']).abs() < 0.06).all()

DuPont Analysis — 2024:
        bank  net_profit_margin  asset_yield  financial_leverage  roe_calculated  roe
  Alpha Bank              29.43         3.08                9.34            8.45  8.5
    Eurobank              43.34         3.69               10.77           17.24 17.2
         NBG              40.14         3.86                9.32           14.43 14.4
Piraeus Bank              38.67         3.52               10.09           13.74 13.7


In [4]:
# Visualize DuPont components
fig = go.Figure()

banks = dupont_2024['bank'].values
x = np.arange(len(banks))
width = 0.25

fig.add_trace(go.Bar(x=x - width, y=dupont_2024['net_profit_margin'], name='Net Profit Margin (%)', marker_color='#3b82f6'))
fig.add_trace(go.Bar(x=x, y=dupont_2024['asset_yield'], name='Asset Yield (%)', marker_color='#10b981'))
fig.add_trace(go.Bar(x=x + width, y=dupont_2024['financial_leverage'], name='Financial Leverage (x)', marker_color='#f59e0b'))

fig.update_layout(
    title='DuPont Analysis — 2024',
    barmode='group',
    xaxis_title='Bank',
    yaxis_title='Value',
    legend_title='Component',
    template='plotly_dark'
)

fig.show()

## 2. Asset Quality

Loan impairments, cost of risk (impairments / average net loans) and the NPE ratio as reported by each bank.

In [5]:
fig = make_subplots(rows=1, cols=3, subplot_titles=('Loan impairments (€m)', 'Cost of risk (% of avg loans)', 'NPE ratio (%)'))

for i, bank in enumerate(df['bank'].unique()):
    bank_data = df[df['bank'] == bank].sort_values('year')
    fig.add_trace(go.Scatter(x=bank_data['year'], y=bank_data['impairment'].abs(), mode='lines+markers', name=bank), row=1, col=1)
    fig.add_trace(go.Scatter(x=bank_data['year'], y=bank_data['cost_of_risk'], mode='lines+markers', name=bank, showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=bank_data['year'], y=bank_data['npe_ratio'], mode='lines+markers', name=bank, showlegend=False), row=1, col=3)

fig.update_layout(title='Asset Quality', template='plotly_dark', height=400)
fig.show()

## 3. Net Interest Margin (NIM) Analysis

In [6]:
# NIM Comparison
fig = go.Figure()

for bank in df['bank'].unique():
    bank_data = df[df['bank'] == bank].sort_values('year')
    fig.add_trace(go.Bar(name=bank, x=bank_data['year'], y=bank_data['nim']))

fig.update_layout(
    title='Net Interest Margin by Bank (%)',
    xaxis_title='Year',
    yaxis_title='NIM (%)',
    barmode='group',
    template='plotly_dark'
)

fig.show()

## 4. Efficiency Analysis (Cost-to-Income)

In [7]:
# Cost-to-Income trends
fig = go.Figure()

for bank in df['bank'].unique():
    bank_data = df[df['bank'] == bank].sort_values('year')
    fig.add_trace(go.Scatter(x=bank_data['year'], y=bank_data['cost_to_income'], mode='lines+markers', name=bank))

fig.update_layout(
    title='Cost-to-Income Ratio (%) — Lower is Better',
    xaxis_title='Year',
    yaxis_title='Cost-to-Income (%)',
    template='plotly_dark'
)

fig.show()

## 5. Peer Comparison Heatmap

In [8]:
# Create comparison matrix for 2024
df_2024 = df[df['year'] == 2024].set_index('bank')

metrics = ['roe', 'nim', 'cost_to_income', 'loan_to_deposit']
metric_labels = {'roe': 'ROE (%)', 'nim': 'NIM (%)', 'cost_to_income': 'Cost-to-Income (%)', 'loan_to_deposit': 'Loan-to-Deposit (%)'}

heatmap_data = df_2024[metrics].T
heatmap_data.index = [metric_labels[m] for m in metrics]

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='RdYlGn',
    text=heatmap_data.values,
    texttemplate='%{z:.1f}',
    textfont={'size': 14}
))

fig.update_layout(title='Bank Comparison Heatmap — 2024', template='plotly_dark', height=350)
fig.show()

## 6. Summary Statistics

In [9]:
# Summary table
summary = df_2024[['nii', 'operating_income', 'net_profit', 'roe', 'nim', 'cost_to_income']].round(1)
summary.columns = ['NII (€m)', 'Op. Income (€m)', 'Net Profit (€m)', 'ROE (%)', 'NIM (%)', 'Cost/Income (%)']

print("📊 2024 Summary:")
print(summary.to_string())

📊 2024 Summary:
              NII (€m)  Op. Income (€m)  Net Profit (€m)  ROE (%)  NIM (%)  Cost/Income (%)
bank                                                                                       
Alpha Bank      1645.0           2222.0            654.0      8.5      2.3             39.0
Eurobank        2507.0           3341.0           1448.0     17.2      2.8             32.9
NBG             2356.0           2885.0           1158.0     14.4      3.2             36.5
Piraeus Bank    2088.0           2757.0           1066.0     13.7      2.7             31.8


---

*Analysis completed. See DATA_DICTIONARY.md for metric definitions.*